# Mini LM Training

## Importing Tools

In [1]:
# ----import tools----

#utilities
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#text processing
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

#LORA Fineting
from peft import LoraConfig, get_peft_model

# Tensorflow and PyTorch
import tensorflow as tf

# CollumnTransformer and Pipeline for preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#evaluation
import evaluate

# MLflow
import mlflow
import mlflow.transformers
from mlflow.tracking import MlflowClient

#Dataset
from datasets import Dataset, Value, Sequence, Features

## Prearing Dataset

In [2]:
# Load Data
training_set_path = "../data/processed/train_ds/data-00000-of-00001.arrow"
ds = Dataset.from_file(training_set_path)
ds.to_pandas().head(10)

,text,labels
0,the original online source for market research...,fraud
1,last up to 4 hours with 100 % natural viagra !...,fraud
2,doctor discovers s ' perm increasement pill\n\...,fraud
3,good time after party : )\n\nthe latest invent...,fraud
4,Important notice: Your account verification re...,fraud
5,Claim a free beauty product! today and enjoy e...,ham
6,FROM 88066 LOST £12 HELP\n,fraud
7,business relationship\n\ni am engineer mr duke...,fraud
8,rolex watches now for pea nut\n\nhows it been ...,fraud
9,"caller: Good afternoon, your catering order fo...",ham


In [3]:
# train test split
ds_split = ds.train_test_split(test_size=0.1, seed=42)

ds_train = ds_split['train']
ds_val = ds_split['test']

print(f"Train Set: {len(ds_train)}, {ds_train.to_pandas()['labels'].value_counts()}")
print(f"Validation Set: {len(ds_val)}, {ds_val.to_pandas()['labels'].value_counts()}")

Train Set: 13307, labels
ham      6704
fraud    6603
Name: count, dtype: int64
Validation Set: 1479, labels
fraud    771
ham      708
Name: count, dtype: int64


## Model URL

In [4]:
minilm_id = "microsoft/Multilingual-MiniLM-L12-H384"

## Tokenizing data

In [5]:
# Decodning {"Fraud":1, "Ham":0}
str2int = {"fraud":1, "ham":0}

encoded_ds_train = ds_train.map(lambda x: {"labels": str2int[x["labels"]]})
encoded_ds_val = ds_val.map(lambda x: {"labels": str2int[x["labels"]]})

# Tokenization for miniLM
tokenizer = AutoTokenizer.from_pretrained(minilm_id)

def tokenize_function(examples):
    return tokenizer(examples["text"],
                      padding="max_length", 
                      truncation=True,
                      max_length=256)



tokenized_ds_train = encoded_ds_train.map(tokenize_function, batched=True)
tokenized_ds_val = encoded_ds_val.map(tokenize_function, batched=True)

print(tokenized_ds_train.features)
print(f"Train Set:{tokenized_ds_train}\n")

print(tokenized_ds_val.features)
print(f"Validation Set:{tokenized_ds_val}\n")

{'text': Value('string'), 'labels': Value('int64'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}
Train Set:Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 13307
})

{'text': Value('string'), 'labels': Value('int64'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}
Validation Set:Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 1479
})



## Configurate Training

In [6]:
from scipy.special import softmax # Standard for NumPy-based metrics
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np

# load model
minilm_model = AutoModelForSequenceClassification.from_pretrained(
    minilm_id, num_labels=2, torch_dtype="float16")

# LORA Config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    bias="none",
    lora_dropout=0.1,
    task_type="SEQ_CLS",
    target_modules=["query", "key", "value", "dense"]
)

# apply LORA
minilm_lora = get_peft_model(minilm_model, lora_config)

# set evaluation metric for LORA finetuning
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predicted_class = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predicted_class)
    precision = precision_score(labels, predicted_class)
    recall = recall_score(labels, predicted_class)
    f1 = f1_score(labels, predicted_class)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
    }


# training arguments
training_args = TrainingArguments(
    output_dir="./results_lora_minilm",
    num_train_epochs=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    save_strategy="best", # save the model at the end of each epoch
    greater_is_better=True, # for recall, higher is better
    load_best_model_at_end=True, # load the best model at the end of training
    metric_for_best_model='recall',
    fp16=False,
    bf16=True,
)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/Multilingual-MiniLM-L12-H384
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Training and Evaluation

In [7]:
from torch import mps
tf.random.set_seed(42)
mps.empty_cache()
# trainer
trainer_minilm_lora = Trainer(
    model=minilm_lora,
    args=training_args,
    train_dataset=tokenized_ds_train,
    eval_dataset=tokenized_ds_val,
    compute_metrics=compute_metrics,
)

# Train the model
trainer_minilm_lora.train()

/opt/miniconda3/envs/dlenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
500,5.290803
1000,1.286773
1500,0.839236
2000,0.609250
2500,0.738050
3000,0.569431


TrainOutput(global_step=3328, training_loss=1.4535325719760015, metrics={'train_runtime': 5742.7124, 'train_samples_per_second': 9.269, 'train_steps_per_second': 0.58, 'total_flos': 1807952937885696.0, 'train_loss': 1.4535325719760015, 'epoch': 4.0})

In [8]:
# Evaluate the model
eval_results = trainer_minilm_lora.evaluate()
print(eval_results)

Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1 Score
0.569431,0.112332,3328,0.979716,0.988142,0.972763,0.980392


{'eval_loss': 0.11233241856098175, 'eval_accuracy': 0.9797160243407708, 'eval_precision': 0.9881422924901185, 'eval_recall': 0.9727626459143969, 'eval_f1_score': 0.9803921568627451}


## LOG Model and Evaluation to ML flow

In [12]:
#set mlflow experiment
mlflow.set_tracking_uri("http://localhost:8080")
mlflow.set_experiment("scam-detector-finetuning")

# log model tokenizer and metrics to mlflow
with mlflow.start_run(run_name="qlora_minilm") as run:
    mlflow.log_params(training_args.to_dict())
    mlflow.log_metrics(eval_results)
    mlflow.transformers.log_model(
        transformers_model= {
            "model": trainer_minilm_lora.model,
            "tokenizer": tokenizer
        },
        artifact_path='qlora_minilm',
        task='text-classification'
    )

# verify if the model is logged in mlflow
client = MlflowClient()
experiment = client.get_experiment_by_name("scam-detector-finetuning")
runs = client.search_runs(experiment_ids=[experiment.experiment_id])
print(runs)

2026/05/10 10:10:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 10:10:54 INFO mlflow.transformers: Overriding save_pretrained to False for PEFT models, following the Transformers behavior. The PEFT adaptor and config will be saved, but the base model weights will not and reference to the HuggingFace Hub repository will be logged instead.
2026/05/10 10:10:55 INFO mlflow.transformers: Skipping saving pretrained model weights to disk as the save_pretrained argumentis set to False. The reference to the HuggingFace Hub repository microsoft/Multilingual-MiniLM-L12-H384 will be logged instead.
2026/05/10 10:10:55 INFO mlflow.transformers: A local checkpoint path or PEFT model is given as the `transformers_model`. To avoid loading the full model into memory, we don't infer the pip requirement for the model. Instead, we will use the default requirements, but it may not capture all required pip libraries for the model. Consider providing the 

🏃 View run qlora_minilm at: http://localhost:8080/#/experiments/1/runs/e4ac9c07da4147f0a5169a16b9735c9b
🧪 View experiment at: http://localhost:8080/#/experiments/1
[<Run: data=<RunData: metrics={'eval_accuracy': 0.9797160243407708,
 'eval_f1_score': 0.9803921568627451,
 'eval_loss': 0.11233241856098175,
 'eval_precision': 0.9881422924901185,
 'eval_recall': 0.9727626459143969}, params={'accelerator_config': "{'split_batches': False, 'dispatch_batches': None, "
                       "'even_batches': True, 'use_seedable_sampler': True, "
                       "'non_blocking': False, 'gradient_accumulation_kwargs': "
                       'None}',
 'adam_beta1': '0.9',
 'adam_beta2': '0.999',
 'adam_epsilon': '1e-08',
 'auto_find_batch_size': 'False',
 'average_tokens_across_devices': 'True',
 'batch_eval_metrics': 'False',
 'bf16': 'True',
 'bf16_full_eval': 'False',
 'data_seed': 'None',
 'dataloader_drop_last': 'False',
 'dataloader_num_workers': '0',
 'dataloader_persistent_workers